# dbt development inside JupyterHub (as *you*)

Do your dbt work right here in the hosted environment — no laptop setup, no shared admin identity. When you logged in, JupyterHub:

- gave you an **editable copy of the dbt project** at `~/dbt` (yours to change),
- injected your **Keycloak identity** and the shared **Spark Connect** endpoint (`SPARK_REMOTE`),
- put a **`uc-dbt`** wrapper on your `PATH`.

`uc-dbt` mints a **Unity Catalog token from your identity** and hands it to dbt, so every model runs **as you** and UC enforces *your* grants. There is no admin/default profile to fall back to.

> **RBAC:** building `silver`/`gold` needs write on those layers (e.g. the `data-engineer` persona — log in as `engineer`). An `analyst` can `compile`/`ls` and read `gold`, but a `build` that writes silver/gold will be **denied** — that's the platform working as designed.

## 1. Who am I? (what can I read/write?)

In [ ]:
import uc_notebook
uc_notebook.whoami()

## 2. How dbt is wired here

- **Project:** `~/dbt` (your editable copy). `profiles.yml` ships inside it.
- **Adapter:** `dbt-spark` with `method: session` — it opens a session on the shared **Spark Connect** server via `SPARK_REMOTE` (same server Dagster + Superset use).
- **Identity:** `profiles.yml` binds `spark.sql.catalog.analytics.token = ${DBT_UC_TOKEN}`. **`uc-dbt` sets `DBT_UC_TOKEN` to your UC token** before every run.

Confirm the connection (runs as you):

In [ ]:
!cd ~/dbt && uc-dbt debug

## 3. Build / run / test — as you

Use `uc-dbt` exactly like `dbt`. From a **JupyterLab terminal** it's the natural workflow:

```bash
cd ~/dbt
uc-dbt build                       # run models + tests
uc-dbt run  --select stg_orders    # a subset
uc-dbt test --select customer_order_summary
```

Or run straight from a notebook cell:

In [ ]:
# List the models dbt sees (cheap, no writes) -- works for any persona.
!cd ~/dbt && uc-dbt ls --resource-type model

In [ ]:
# Build silver + gold. Requires WRITE on those layers (data-engineer / engineer).
# As an analyst this is DENIED by Unity Catalog -- that denial is expected.
!cd ~/dbt && uc-dbt build --select stg_orders stg_customers customer_order_summary

## 4. Inspect the result through Spark (still as you)

In [ ]:
spark = uc_notebook.uc_session()
spark.sql("SELECT * FROM analytics.gold.customer_order_summary ORDER BY completed_orders DESC LIMIT 5").show()

## 5. Editing models

Everything in `~/dbt` is **yours to edit** — open `models/…/*.sql` in the file browser, change them, and re-run `uc-dbt build`. Your copy is isolated and persisted across sessions.

**What this maps to in production:** the *same* dbt project is orchestrated by Dagster, where it runs as a team's **build service account** (`sa-team-<team>-build`) rather than an interactive user. Identity is the only difference — the models, `profiles.yml`, Spark Connect path, and UC governance are identical. Promote your changes by committing them back to the shared project.